# NB03 -- Tabular Survival Benchmarks (v1.0-ram Multi-Cohort)
**Stage:** NB03 | **RAM strategy:** sequential cohorts, float32, del+gc per cohort

### What this notebook does
Leakage-free tabular survival baselines on raw expression across all three TCGA
cohorts (LGG, KIRC, LUAD). Two Cox-MLP tracks compared head-to-head:

| Track | Architecture | Purpose |
|-------|-------------|---------|
| **Option A** | Deep, highly regularised | Honest, stable baseline |
| **Option B** | Shallow / wide | Peak-validation reference |

**RAM strategy (mirrors NB00/NB02/NB04):**

| Technique | Where |
|-----------|-------|
| `del data / model / opt` + `gc.collect()` after each cohort | Cell 7 |
| `workers=0` in DataLoader | Cell 5 |
| State dict stored `.cpu().clone()` only | Cell 5 |
| Sequential cohort loop -- only one cohort in RAM at a time | Cell 7 |
| Figures closed with `plt.close()` immediately after saving | Cell 3 |
| Full model **not** kept in `ALL_RESULTS` after export | Cell 7 |

**Reads (from NB02 checkpoints):**
- `checkpoints/NB02/NB02_{cohort}_dae_discrete.pt`

**Writes (canonical NB03 layout):**
- `results/NB03/figures/NB03_{cohort}_*.png`
- `results/NB03/NB03_benchmark_summary.csv`


### Cell 1 -- Imports & Seeding
Locks all RNG states. Sets `NB_VERSION` and `PIPELINE_STAGE`.

In [1]:
# ==============================================================================
# CELL 1: IMPORTS & DETERMINISTIC SEEDING
# ==============================================================================
import os, gc, random, warnings, platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # no GUI -- saves RAM on headless machines
import matplotlib.pyplot as plt
import yaml, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore")

NB_VERSION     = "1.0-ram"
PIPELINE_STAGE = "NB03"
SEED           = 42

def enforce_determinism(seed: int = 42) -> None:
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

enforce_determinism(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\u2705 {PIPELINE_STAGE} v{NB_VERSION} | Device: {device} | Seed: {SEED}")
print(f"   Python  : {platform.python_version()}")
print(f"   PyTorch : {torch.__version__}")


✅ NB03 v1.0-ram | Device: cpu | Seed: 42
   Python  : 3.12.3
   PyTorch : 2.12.0+cpu


### Cell 2 -- Config & Canonical Paths
Loads `config.yaml`, NB00/NB02 checkpoint refs, builds NB03 output dirs.

In [2]:
# ==============================================================================
# CELL 2: CONFIG & CANONICAL PATHS  (NB03 layout)
# ==============================================================================
CONFIG_PATH = Path("config.yaml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError("config.yaml missing from workspace root.")

with open(CONFIG_PATH, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

BASE_DIR = Path.cwd()
pp  = cfg.get("preprocessing", {})
m   = cfg.get("model",         {})
t   = cfg.get("training",      {})
nb3 = cfg.get("nb03",          {})

# ── Hyperparameters -- all from config, NB03 section overrides globals ────────
TARGET_FEATURES = pp.get("variance_filter_top_n",             3000)
BATCH_SIZE      = t.get("batch_size",                           64)
MAX_EPOCHS      = t.get("epochs",                               40)
PATIENCE        = nb3.get("patience",  t.get("patience",        12))
LR_A            = nb3.get("lr_optA",                          5e-5)
WD_A            = nb3.get("wd_optA",                          0.04)
LR_B            = nb3.get("lr_optB",                          1e-4)
WD_B            = nb3.get("wd_optB",                          0.02)
NB00_VERSION    = nb3.get("nb00_version", "3_3_ram")   # matches NB00 NB_VERSION

# ── Canonical output dirs ─────────────────────────────────────────────────────
out_cfg        = cfg.get("output", {})
_ckpt_root     = out_cfg.get("checkpoint_dir", "checkpoints")
_results_root  = out_cfg.get("results_dir",    "results")
_figures_sub   = out_cfg.get("figures_subdir", "figures")

NB02_CKPT_DIR  = BASE_DIR / _ckpt_root / "NB02"
CHECKPOINT_DIR = BASE_DIR / _ckpt_root / PIPELINE_STAGE
FIGURES_DIR    = BASE_DIR / _results_root / PIPELINE_STAGE / _figures_sub
RESULTS_DIR    = BASE_DIR / _results_root / PIPELINE_STAGE
EMBEDDINGS_DIR = BASE_DIR / out_cfg.get("embeddings_dir", "embeddings") / PIPELINE_STAGE
RAW_DIR        = BASE_DIR / cfg.get("data", {}).get("raw_dir", "data/raw")

for _d in [CHECKPOINT_DIR, FIGURES_DIR, EMBEDDINGS_DIR]: _d.mkdir(parents=True, exist_ok=True)

# ── Multi-cohort config -- driven entirely from config.yaml cohorts block ─────
COHORT_CONFIGS = []
for entry in cfg.get("cohorts", []):
    COHORT_CONFIGS.append({
        "name":  entry["name"],
        "short": entry["short"],
        "label": entry.get("label", entry["name"]),
        "expr":  RAW_DIR / entry["expression_file"],
        "surv":  RAW_DIR / entry["survival_file"],
    })

if not COHORT_CONFIGS:
    raise ValueError("No cohorts defined in config.yaml -- add a `cohorts:` block.")

# ── NB00 baseline loader -- version string from config ────────────────────────
NB00_CI = {}
for cc in COHORT_CONFIGS:
    sh = cc["short"].upper()
    p  = BASE_DIR / _ckpt_root / "NB00" / f"NB00_{sh}_v{NB00_VERSION}_mlp_baseline.pt"
    if p.exists():
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            NB00_CI[cc["short"]] = ck.get("honest_ci", {}).get("c_index")
            ci_s = f"{NB00_CI[cc['short']]:.4f}" if NB00_CI[cc["short"]] else "n/a"
            print(f"   NB00 [{sh}]: {ci_s}")
        except Exception as e:
            print(f"   NB00 [{sh}]: load error -- {e}")
            NB00_CI[cc["short"]] = None
    else:
        NB00_CI[cc["short"]] = None
        print(f"   NB00 [{sh}]: checkpoint not found (run NB00 first)")

print(f"\n\U0001f680 {PIPELINE_STAGE} v{NB_VERSION}  |  {cfg['project']['name']} v{cfg['project']['version']}")
print(f"   Checkpoints : {_ckpt_root}/{PIPELINE_STAGE}/")
print(f"   Figures     : {_results_root}/{PIPELINE_STAGE}/{_figures_sub}/")
print(f"   Embeddings  : {out_cfg.get('embeddings_dir','embeddings')}/{PIPELINE_STAGE}/")
print(f"   Cohorts     : {[c['short'] for c in COHORT_CONFIGS]}")
print(f"   OptA lr={LR_A} wd={WD_A}  |  OptB lr={LR_B} wd={WD_B}  |  patience={PATIENCE}")
print(f"   NB00 version key: v{NB00_VERSION}")


   NB00 [LGG]: checkpoint not found (run NB00 first)
   NB00 [KIRC]: checkpoint not found (run NB00 first)
   NB00 [LUAD]: checkpoint not found (run NB00 first)

🚀 NB03 v1.0-ram  |  universal-survival-engine v3.3
   Checkpoints : checkpoints/NB03/
   Figures     : results/NB03/figures/
   Embeddings  : embeddings/NB03/
   Cohorts     : ['lgg', 'kirc', 'luad']
   OptA lr=5e-05 wd=0.04  |  OptB lr=0.0001 wd=0.02  |  patience=12
   NB00 version key: v3_3_ram


### Cell 3 -- Dataset, Architectures, Losses, Utilities
All class/function definitions. No data loaded here.

In [3]:
# ==============================================================================
# CELL 3: DATASET, ARCHITECTURES, LOSSES, UTILITIES
# ==============================================================================

class SurvivalDataset(Dataset):
    def __init__(self, X, times, events):
        self.X = torch.tensor(X,      dtype=torch.float32)
        self.t = torch.tensor(times,  dtype=torch.float32)
        self.e = torch.tensor(events, dtype=torch.float32)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.t[i], self.e[i]

def cox_partial_loss(risk, times, events):
    order      = torch.argsort(times, descending=True)
    risk_s     = risk[order]; events_s = events[order]
    log_cumsum = torch.logcumsumexp(risk_s, dim=0)
    nll        = -(risk_s - log_cumsum) * events_s
    return nll.sum() / events_s.sum().clamp(min=1.0)

class DeepCoxMLP(nn.Module):
    "Option A -- Deep, highly regularised. Stable generalisation baseline."
    def __init__(self, input_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512,       128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,        64), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.LayerNorm(64),
        )
        self.head = nn.Linear(64, 1, bias=False)
    def forward(self, x):
        z = self.encoder(x)
        return self.head(z).squeeze(-1)
    def encode(self, x):
        """Return 64-dim latent for embedding export."""
        return self.encoder(x)

class ShallowCoxMLP(nn.Module):
    "Option B -- Shallow / wide. Higher validation ceiling, less stable."
    def __init__(self, input_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256,        64), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.LayerNorm(64),
        )
        self.head = nn.Linear(64, 1, bias=False)
    def forward(self, x):
        z = self.encoder(x)
        return self.head(z).squeeze(-1)
    def encode(self, x):
        """Return 64-dim latent for embedding export."""
        return self.encoder(x)

def plot_km(risk, times, events, title, fpath, thresh=None):
    thresh = thresh if thresh is not None else np.median(risk)
    hi, lo = risk >= thresh, risk < thresh
    fig, ax = plt.subplots(figsize=(8, 5))
    for mask, label, color in [
        (hi, f"High Risk (n={hi.sum()})", "crimson"),
        (lo, f"Low Risk  (n={lo.sum()})", "dodgerblue"),
    ]:
        if mask.sum() > 1:
            KaplanMeierFitter().fit(times[mask], events[mask], label=label
                                   ).plot_survival_function(ax=ax, ci_show=True, color=color)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Days"); ax.set_ylabel("Survival Probability")
    ax.grid(True, linestyle=":", alpha=0.5); ax.legend(loc="lower left")
    fig.tight_layout(); fig.savefig(fpath, dpi=150, bbox_inches="tight")
    plt.close(); gc.collect()
    print(f"   \U0001f4ca {Path(fpath).name}")

def bootstrap_ci(times, events, risks, label, n_boot=1000):
    mask = np.isfinite(times) & np.isfinite(events) & np.isfinite(risks)
    times, events, risks = times[mask], events[mask], risks[mask]
    if len(times) == 0 or events.sum() == 0:
        print(f"   [{label}] No valid samples."); return None
    ci = concordance_index(times, -risks, events)
    rng = np.random.default_rng(SEED); boot = []
    idx = np.arange(len(times))
    for _ in range(n_boot):
        bi = rng.choice(idx, size=len(times), replace=True)
        if events[bi].sum() == 0: continue
        try: boot.append(concordance_index(times[bi], -risks[bi], events[bi]))
        except: continue
    if boot:
        lo, hi = np.percentile(boot, [2.5, 97.5]); sd = np.std(boot)
        print(f"   [{label}] C-Index: {ci:.4f} | 95% CI [{lo:.4f}-{hi:.4f}] | SD +-{sd:.4f}")
        return {"c_index": ci, "lo": lo, "hi": hi, "sd": sd}
    print(f"   [{label}] C-Index: {ci:.4f}  (bootstrap failed)")
    return {"c_index": ci, "lo": None, "hi": None, "sd": None}

print("\u2705 SurvivalDataset, DeepCoxMLP, ShallowCoxMLP, cox_partial_loss, plot_km, bootstrap_ci defined.")
print("   Both models expose .encode(x) -> 64-dim latent for embedding export.")


✅ SurvivalDataset, DeepCoxMLP, ShallowCoxMLP, cox_partial_loss, plot_km, bootstrap_ci defined.
   Both models expose .encode(x) -> 64-dim latent for embedding export.


### Cell 4 -- Per-Cohort Data Loader
Variance-filters raw expression, splits 60/20/20. Called once per cohort inside the main loop.

In [4]:
# ==============================================================================
# CELL 4: PER-COHORT DATA LOADER  (mirrors NB02 load_cohort exactly)
# RAM: float32 on load, nlargest variance filter, early dropna, valid mask.
# ==============================================================================
def _trim(s):
    s = str(s).strip().replace(".", "-").upper()
    return "-".join(s.split("-")[:3]) if s.startswith("TCGA") else s

def _strip_v(ensg): return str(ensg).split(".")[0]

def load_cohort(cc: dict, top_n: int = TARGET_FEATURES) -> dict:
    """RAM-optimised TCGA loader (float32, nlargest variance, early drop).
    Prints the NB02-style: 📂 name | N patients | G genes | E events | RAM | dropped N neg-time
    """
    name = cc["name"]; label = cc.get("label", name)
    print(f"  \U0001f4c2 {name}")

    # Survival -- read only needed columns
    clin = pd.read_csv(cc["surv"], sep="\t", index_col=0)
    clin.index = pd.Index([_trim(i) for i in clin.index])
    clin = clin[~clin.index.duplicated(keep="first")]
    time_col  = next((c for c in ["OS.time","survival_time","time"]  if c in clin.columns), None)
    event_col = next((c for c in ["OS","event","status"]             if c in clin.columns), None)
    if not time_col or not event_col:
        raise KeyError(f"[{name}] No time/event columns in {list(clin.columns)}")

    # Expression -- float32 on load, variance filter immediately
    df = pd.read_csv(cc["expr"], sep="\t", index_col=0)
    if df.shape[0] > df.shape[1]: df = df.T
    df.columns = pd.Index([_strip_v(c) for c in df.columns])
    df = df.loc[:, ~df.columns.duplicated(keep="first")]
    df.index = pd.Index([_trim(i) for i in df.index])
    df = df[~df.index.duplicated(keep="first")].astype(np.float32)

    # Variance filter via nlargest -- never sorts full array
    top_genes = df.var(axis=0, ddof=0).nlargest(min(top_n, df.shape[1])).index.tolist()
    df = df[top_genes]

    # Align, filter, clean
    common = df.index.intersection(clin.index)
    df     = df.loc[common]
    surv   = clin.loc[common, [time_col, event_col]].copy()
    surv.columns = ["t", "e"]
    valid  = surv["t"].notna() & surv["e"].notna() & (surv["t"] > 0)
    df     = df.loc[valid].dropna(axis=1).astype(np.float32)
    surv   = surv.loc[valid].astype(np.float32)
    del clin; gc.collect()

    y_time  = surv["t"].values.astype(np.float32)
    y_event = surv["e"].values.astype(np.float32)

    ram_mb = df.memory_usage(deep=True).sum() / 1e6
    n_drop = int((~valid).sum())
    print(f"     {len(df)} patients | {df.shape[1]:,} genes | "
          f"{int(y_event.sum())} events ({100*y_event.mean():.1f}%) | "
          f"RAM: {ram_mb:.1f} MB" + (f" | dropped {n_drop} neg-time" if n_drop else ""))

    # Stratified 60/20/20 split
    X_raw = df.values.astype(np.float32)
    del df; gc.collect()

    X_tr_r, X_te_r, t_tr, t_te, e_tr, e_te = train_test_split(
        X_raw, y_time, y_event,
        test_size=0.20, stratify=y_event.astype(int), random_state=SEED)
    X_tr_r, X_va_r, t_tr, t_va, e_tr, e_va = train_test_split(
        X_tr_r, t_tr, e_tr,
        test_size=0.25, stratify=e_tr.astype(int), random_state=SEED)

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_tr_r).astype("float32")
    X_val   = scaler.transform(X_va_r).astype("float32")
    X_test  = scaler.transform(X_te_r).astype("float32")
    del X_tr_r, X_va_r, X_te_r; gc.collect()

    return dict(
        name=name, label=label,
        X_train=X_train, X_val=X_val, X_test=X_test,
        t_tr=t_tr, t_va=t_va, t_te=t_te,
        e_tr=e_tr, e_va=e_va, e_te=e_te,
        X_raw=X_raw, y_time=y_time, y_event=y_event,
        input_dim=X_train.shape[1], scaler=scaler,
    )

print("\u2705 load_cohort() defined (NB02-identical: float32, nlargest, dropna, valid mask).")


✅ load_cohort() defined (NB02-identical: float32, nlargest, dropna, valid mask).


### Cell 5 -- Leakage-Free Training Engine
Early stopping on val C-index. Best state restored before test eval. `workers=0` throughout.

In [5]:
# ==============================================================================
# CELL 5: LEAKAGE-FREE TRAINING ENGINE
# ==============================================================================
def _safe_cindex(times, risks, events):
    """concordance_index wrapper -- returns 0.5 on degenerate inputs instead of crashing."""
    try:
        mask = np.isfinite(times) & np.isfinite(risks) & np.isfinite(events)
        if mask.sum() < 2 or events[mask].sum() == 0:
            return 0.5
        # lifelines: higher predicted = better survival -> negate risk scores
        ci = concordance_index(times[mask], -risks[mask], events[mask])
        return float(ci) if np.isfinite(ci) else 0.5
    except Exception:
        return 0.5

def train_cox_pipeline(
    model, X_train, X_val, X_test,
    t_tr, t_va, t_te, e_tr, e_va, e_te,
    optimizer, run_label, cname, sh,
    max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, patience=PATIENCE,
) -> dict:
    # drop_last=False -- never silently discard events from small val splits
    train_ldr = DataLoader(
        SurvivalDataset(X_train, t_tr, e_tr),
        batch_size=batch_size, shuffle=True, drop_last=False, num_workers=0)

    best_val_c = -1.0; best_state = None; wait = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        ep_loss, n = 0.0, 0
        for bx, bt, be in train_ldr:
            bx = bx.to(device); bt = bt.to(device); be = be.to(device)
            # skip degenerate batches (no events -- cox loss undefined)
            if be.sum() == 0: continue
            optimizer.zero_grad()
            loss = cox_partial_loss(model(bx), bt, be)
            if not torch.isfinite(loss): continue
            loss.backward(); optimizer.step()
            ep_loss += loss.item() * bx.size(0); n += bx.size(0)

        if n == 0: continue   # entire epoch had no valid batches

        model.eval()
        with torch.no_grad():
            val_risk = model(torch.tensor(X_val, dtype=torch.float32).to(device)).cpu().numpy().ravel()
        val_c = _safe_cindex(t_va, val_risk, e_va)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  [{run_label}] Ep {epoch:3d}/{max_epochs}"
                  f" | Loss {ep_loss/n:.4f} | Val C {val_c:.4f}")

        if val_c > best_val_c:
            best_val_c = val_c
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {epoch}.")
                break

    if best_state is None:
        print(f"  WARNING [{run_label}]: no valid epoch -- returning untrained model.")
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_risk  = model(torch.tensor(X_test,  dtype=torch.float32).to(device)).cpu().numpy().ravel()
        train_risk = model(torch.tensor(X_train, dtype=torch.float32).to(device)).cpu().numpy().ravel()

    test_c       = _safe_cindex(t_te, test_risk, e_te)
    train_thresh = np.median(train_risk)
    del train_risk; gc.collect()

    ci_result = bootstrap_ci(t_te, e_te, test_risk, f"{run_label} [{sh.upper()}]")
    print(f"  Best Val C: {best_val_c:.4f}  |  Test C: {test_c:.4f}")

    plot_km(test_risk, t_te, e_te,
            title=f"{run_label} -- {cname} Test KM",
            fpath=FIGURES_DIR / f"NB03_{sh}_{run_label.lower().replace(' ','_')}_km.png",
            thresh=train_thresh)
    return {"val_c": best_val_c, "test_c": test_c, "honest_ci": ci_result}

print("\u2705 train_cox_pipeline() defined.")


✅ train_cox_pipeline() defined.


### Cell 6 -- Dead Toll & KM Milestone Table
Mirrors NB04 Cell 9b exactly. Prints per-group Y1/Y3/Y5 survival statistics.

In [6]:
# ==============================================================================
# CELL 6: DEAD TOLL & KM MILESTONE TABLE FUNCTION  (mirrors NB04 Cell 9b)
# ==============================================================================
_milestone_years = cfg.get("nb03", {}).get("km_milestones_years", [1, 3, 5])
MILESTONES_DAYS  = [y * 365 for y in _milestone_years]

def print_dead_toll_table(cname, y_time, y_event, risk_scores):
    """NB02/NB04-identical KM milestone table. Inputs must already be valid (no NaN/neg)."""
    y_time      = np.asarray(y_time,      dtype="float32").ravel()
    y_event     = np.asarray(y_event,     dtype="float32").ravel()
    risk_scores = np.asarray(risk_scores, dtype="float32").ravel()

    dead_toll      = int(y_event.sum())
    survival_count = int((1 - y_event).sum())
    n_total        = len(y_event)

    print(f"\n  [{cname}]  N={n_total}")
    print(f"    Dead Toll      (\u03a3 \u03b4\u1d35 = 1) : {dead_toll}  ({100*dead_toll/n_total:.1f}%)")
    print(f"    Survival Count (\u03a3 1-\u03b4\u1d35)   : {survival_count}  ({100*survival_count/n_total:.1f}%)")

    thresh = np.median(risk_scores)
    hi_m   = risk_scores >= thresh
    lo_m   = ~hi_m

    rows = []
    for group_label, mask_g in [("High-Risk", hi_m), ("Low-Risk", lo_m),
                                  ("Overall", np.ones(n_total, bool))]:
        t_g = y_time[mask_g]; e_g = y_event[mask_g]
        if len(t_g) == 0: continue
        kmf = KaplanMeierFitter(label=group_label)
        kmf.fit(t_g, e_g)
        et  = kmf.event_table
        for ms in MILESTONES_DAYS:
            yr  = ms // 365
            sub = et[et.index <= ms]
            at_risk    = int(sub["at_risk"].iloc[-1])  if len(sub) else int(mask_g.sum())
            n_events   = int(sub["observed"].sum())
            n_censored = int(sub["censored"].sum())
            sf_val     = float(kmf.survival_function_at_times([ms]).iloc[0])
            rows.append({"Cohort": cname, "Group": group_label, "Year": f"Y{yr}",
                         "At-Risk": at_risk, "Events (Dead)": n_events,
                         "Censored": n_censored, "S(t)": round(sf_val, 4)})

    tbl = pd.DataFrame(rows)
    print()
    print(tbl.to_string(index=False))
    return tbl

print("\u2705 print_dead_toll_table() defined.")


✅ print_dead_toll_table() defined.


### Cell 7 -- Main Cohort Loop
One cohort at a time: load -> train A -> train B -> dead toll table -> del+gc.

In [7]:
# ==============================================================================
# CELL 7: MAIN COHORT LOOP
# RAM: one cohort in memory at a time; full del+gc between cohorts.
# ==============================================================================
ALL_RESULTS  = {}   # lightweight summaries only -- no tensors kept
ALL_TOLL_TBL = []   # populated in Cell 8 (dead toll) after training

for cc in COHORT_CONFIGS:
    cname = cc["name"]; sh = cc["short"]
    print(f"\n{'='*65}")
    print(f"Cohort: {cname}  ({cc['label']})")
    print(f"{'='*65}")

    # A. Load data (📂 header + stats printed inside load_cohort)
    data = load_cohort(cc)
    INPUT_DIM = data["input_dim"]

    # B. Option A -- Deep regularised
    enforce_determinism(SEED)
    model_a = DeepCoxMLP(INPUT_DIM).to(device)
    opt_a   = optim.AdamW(model_a.parameters(), lr=LR_A, weight_decay=WD_A)
    print(f"\nOption A -- Deep [{cname}]")
    res_a   = train_cox_pipeline(
        model_a, data["X_train"], data["X_val"], data["X_test"],
        data["t_tr"], data["t_va"], data["t_te"],
        data["e_tr"], data["e_va"], data["e_te"],
        opt_a, "optA_deep", cname, sh,
    )
    # Save best model state for dead toll cell
    torch.save({
        "model_state": {k: v.cpu().clone() for k, v in model_a.state_dict().items()},
        "input_dim": INPUT_DIM,
        "option": "A",
        "honest_ci": res_a.get("honest_ci"),
    }, CHECKPOINT_DIR / f"NB03_{sh}_optA_deep.pt")
    del model_a, opt_a; gc.collect()

    # C. Option B -- Shallow / wide
    enforce_determinism(SEED)
    model_b = ShallowCoxMLP(INPUT_DIM).to(device)
    opt_b   = optim.AdamW(model_b.parameters(), lr=LR_B, weight_decay=WD_B)
    print(f"\nOption B -- Shallow [{cname}]")
    res_b   = train_cox_pipeline(
        model_b, data["X_train"], data["X_val"], data["X_test"],
        data["t_tr"], data["t_va"], data["t_te"],
        data["e_tr"], data["e_va"], data["e_te"],
        opt_b, "optB_shallow", cname, sh,
    )
    torch.save({
        "model_state": {k: v.cpu().clone() for k, v in model_b.state_dict().items()},
        "input_dim": INPUT_DIM,
        "option": "B",
        "honest_ci": res_b.get("honest_ci"),
    }, CHECKPOINT_DIR / f"NB03_{sh}_optB_shallow.pt")
    del model_b, opt_b; gc.collect()

    # D. Lightweight summary -- no tensors
    ALL_RESULTS[sh] = {
        "cohort": cname, "opt_a": res_a, "opt_b": res_b,
        "nb00_ci": NB00_CI.get(sh),
    }

    # ── E. Embedding export ───────────────────────────────────────────────
    z_cols = [f"z_{i}" for i in range(64)]
    for opt_tag, ckpt_file in [("optA", f"NB03_{sh}_optA_deep.pt"),
                               ("optB", f"NB03_{sh}_optB_shallow.pt")]:
        ck   = torch.load(CHECKPOINT_DIR / ckpt_file, weights_only=False)
        _cls = DeepCoxMLP if opt_tag == "optA" else ShallowCoxMLP
        _m   = _cls(ck["input_dim"]).to(device)
        _m.load_state_dict(ck["model_state"]); _m.eval()

        # — held-out: scaler from train split (leakage-safe) —
        X_ho = data["scaler"].transform(data["X_raw"]).astype(np.float32)
        with torch.no_grad():
            z_ho   = _m.encode(torch.tensor(X_ho).to(device)).cpu().numpy()
            risk_ho = _m(torch.tensor(X_ho).to(device)).cpu().numpy()
        df_ho = pd.DataFrame(z_ho, columns=z_cols)
        df_ho["risk_score"] = risk_ho
        df_ho["survival_time"] = data["y_time"]; df_ho["event"] = data["y_event"]
        ho_path = EMBEDDINGS_DIR / f"NB03_{sh}_{opt_tag}_heldout_latents.csv"
        df_ho.to_csv(ho_path, index=False)
        print(f"  📁 {ho_path.name}  (leakage-safe — train scaler)")
        del X_ho, z_ho, risk_ho, df_ho

        # — full-cohort: scaler refit on all patients (representation only) —
        sc_full  = StandardScaler()
        X_fc     = sc_full.fit_transform(data["X_raw"]).astype(np.float32)
        with torch.no_grad():
            z_fc    = _m.encode(torch.tensor(X_fc).to(device)).cpu().numpy()
            risk_fc = _m(torch.tensor(X_fc).to(device)).cpu().numpy()
        del X_fc
        df_fc = pd.DataFrame(z_fc, columns=z_cols)
        df_fc["risk_score"] = risk_fc
        df_fc["survival_time"] = data["y_time"]; df_fc["event"] = data["y_event"]
        fc_path = EMBEDDINGS_DIR / f"NB03_{sh}_{opt_tag}_fullcohort_latents.csv"
        df_fc.to_csv(fc_path, index=False)
        print(f"  📁 {fc_path.name}  (full-cohort scaler — representation only)")
        del _m, z_fc, risk_fc, df_fc, sc_full
        gc.collect()

    del data; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"  RAM released [{cname}].")

print(f"\n{'='*65}")
print(f"\u2705 NB03 training complete -- {len(ALL_RESULTS)} cohorts. Run Cell 8 for dead toll table.")



Cohort: TCGA-LGG  (Brain LGG)
  📂 TCGA-LGG
     512 patients | 3,000 genes | 126 events (24.6%) | RAM: 6.2 MB | dropped 4 neg-time

Option A -- Deep [TCGA-LGG]
  [optA_deep] Ep   1/40 | Loss 3.2306 | Val C 0.7100
  [optA_deep] Ep  10/40 | Loss 2.1608 | Val C 0.8351
  [optA_deep] Ep  20/40 | Loss 1.9357 | Val C 0.8218
  Early stopping at epoch 22.
   [optA_deep [LGG]] C-Index: 0.8240 | 95% CI [0.7437-0.8930] | SD +-0.0378
  Best Val C: 0.8351  |  Test C: 0.8240
   📊 NB03_lgg_opta_deep_km.png

Option B -- Shallow [TCGA-LGG]
  [optB_shallow] Ep   1/40 | Loss 2.8224 | Val C 0.7308
  [optB_shallow] Ep  10/40 | Loss 1.7921 | Val C 0.8019
  [optB_shallow] Ep  20/40 | Loss 1.4890 | Val C 0.8237
  [optB_shallow] Ep  30/40 | Loss 1.3436 | Val C 0.8246
  Early stopping at epoch 38.
   [optB_shallow [LGG]] C-Index: 0.8437 | 95% CI [0.7836-0.8999] | SD +-0.0316
  Best Val C: 0.8322  |  Test C: 0.8437
   📊 NB03_lgg_optb_shallow_km.png
  📁 NB03_lgg_optA_heldout_latents.csv  (leakage-safe — train sca

### Cell 8 — Dead Toll & Survival Table
Runs after Cell 7. Reloads each cohort fresh + reloads checkpoint. Mirrors NB02 Cell 6b exactly.

In [8]:
# ==============================================================================
# CELL 8: DEAD TOLL + SURVIVAL COUNT + KM MILESTONE TABLE
# Mirrors NB02 Cell 6b exactly: reloads cohort + checkpoint after training loop.
# Must run after Cell 7 (ALL_RESULTS populated, checkpoints saved).
# ==============================================================================
print(f"\n{'='*72}")
print("  DEAD TOLL & SURVIVAL TABLE \u2014 NB03 Tabular Benchmark")
print(f"{'='*72}")

for cc in COHORT_CONFIGS:
    name = cc["name"]; sh = cc["short"]
    ckpt_p = CHECKPOINT_DIR / f"NB03_{sh}_optA_deep.pt"
    if not ckpt_p.exists():
        print(f"  [{name}] checkpoint not found -- skipping (run Cell 7 first)"); continue

    # A. Reload full cohort (📂 header printed inside load_cohort)
    cohort  = load_cohort(cc)
    y_time  = cohort["y_time"]
    y_event = cohort["y_event"]
    X_raw   = cohort["X_raw"]

    # B. Reload Option A checkpoint and re-derive risk scores
    ck = torch.load(ckpt_p, map_location="cpu", weights_only=False)
    _m = DeepCoxMLP(ck["input_dim"]).to(device)
    _m.load_state_dict(ck["model_state"]); _m.eval()

    # Scale full cohort with a fresh scaler (display only -- not training leakage)
    sc = StandardScaler(); sc.fit(X_raw)
    X_sc = sc.transform(X_raw).astype("float32")
    with torch.no_grad():
        risk = _m(torch.tensor(X_sc).to(device)).cpu().numpy().ravel()
    del _m, X_sc, sc; gc.collect()

    # C. Print dead toll + KM milestone table
    ALL_TOLL_TBL.append(
        print_dead_toll_table(name, y_time, y_event, risk)
    )

    del cohort, X_raw, risk; gc.collect()

print(f"\n{'='*72}")



  DEAD TOLL & SURVIVAL TABLE — NB03 Tabular Benchmark
  📂 TCGA-LGG
     512 patients | 3,000 genes | 126 events (24.6%) | RAM: 6.2 MB | dropped 4 neg-time

  [TCGA-LGG]  N=512
    Dead Toll      (Σ δᴵ = 1) : 126  (24.6%)
    Survival Count (Σ 1-δᴵ)   : 386  (75.4%)

  Cohort     Group Year  At-Risk  Events (Dead)  Censored   S(t)
TCGA-LGG High-Risk   Y1      189             26        43 0.8846
TCGA-LGG High-Risk   Y3       67             74       116 0.5702
TCGA-LGG High-Risk   Y5       30             90       137 0.3940
TCGA-LGG  Low-Risk   Y1      209              0        48 1.0000
TCGA-LGG  Low-Risk   Y3       93              1       163 0.9948
TCGA-LGG  Low-Risk   Y5       38              8       211 0.8895
TCGA-LGG   Overall   Y1      397             26        91 0.9419
TCGA-LGG   Overall   Y3      159             75       279 0.7771
TCGA-LGG   Overall   Y5       67             98       348 0.6245
  📂 TCGA-KIRC
     531 patients | 3,000 genes | 175 events (33.0%) | RAM: 6.4 MB



### Cell 9 — Summary & NB00 Delta
Mirrors NB04 Cell 12. Per-cohort table + delta vs NB00 baseline. Saves CSV.

In [9]:
# ==============================================================================
# CELL 8: SUMMARY TABLE & NB00 DELTA  (mirrors NB04 Cell 12)
# Requires: ALL_RESULTS populated (Cell 7 complete).
# ==============================================================================
print("=" * 65)
print(f"NB03 v{NB_VERSION} -- TABULAR SURVIVAL BENCHMARK SUMMARY")
print("=" * 65)
print()
print(f"  {'Cohort':<12} {'Option':>12} {'Val C':>8} {'Test C':>8} {'Gap':>7} {'Flag':>5}")
print("  " + "-" * 54)

summary_rows = []
for sh, r in ALL_RESULTS.items():
    for opt_key, opt_label in [("opt_a", "DeepMLP-A"), ("opt_b", "ShallowMLP-B")]:
        vc = r[opt_key]["val_c"]; tc = r[opt_key]["test_c"]
        gap  = abs(vc - tc)
        flag = "OK" if gap < 0.03 else ("WATCH" if gap < 0.08 else "LEAK")
        print(f"  {r['cohort']:<12} {opt_label:>12} {vc:>8.4f} {tc:>8.4f} {gap:>7.4f} {flag:>6}")
        summary_rows.append({"cohort": r["cohort"], "option": opt_label,
                              "val_c": round(vc, 4), "test_c": round(tc, 4),
                              "gap": round(gap, 4)})

print("=" * 65)
print("  OK gap < 0.03 = stable | WATCH gap < 0.08 | LEAK gap >= 0.08 = risk")

print()
print(f"  NB00 -> NB03 delta (Option A test C-index)")
print(f"  {'Cohort':<12} {'NB00':>8}  {'NB03-A':>8}  {'Delta':>8}")
print("  " + "-" * 44)
for sh, r in ALL_RESULTS.items():
    nb0 = r["nb00_ci"]; nb3 = r["opt_a"]["test_c"]
    delta = f"{nb3 - nb0:+.4f}" if nb0 else "n/a"
    flag  = "UP" if nb0 and nb3 > nb0 else ("DOWN" if nb0 else "--")
    nb0_s = f"{nb0:.4f}" if nb0 else "n/a"
    print(f"  {r['cohort']:<12} {nb0_s:>8}  {nb3:>8.4f}  {delta:>8} {flag}")

print("=" * 65)

summary_df = pd.DataFrame(summary_rows)
out_csv    = RESULTS_DIR / "NB03_benchmark_summary.csv"
summary_df.to_csv(out_csv, index=False)
print(f"\n  Saved: {out_csv}")
print("\n  Outputs:")
for sh in ALL_RESULTS:
    print(f"    results/NB03/figures/NB03_{sh}_optA_deep_km.png")
    print(f"    results/NB03/figures/NB03_{sh}_optB_shallow_km.png")
print(f"    results/NB03/NB03_benchmark_summary.csv")
print("=" * 65)


NB03 v1.0-ram -- TABULAR SURVIVAL BENCHMARK SUMMARY

  Cohort             Option    Val C   Test C     Gap  Flag
  ------------------------------------------------------
  TCGA-LGG        DeepMLP-A   0.8351   0.8240  0.0111     OK
  TCGA-LGG     ShallowMLP-B   0.8322   0.8437  0.0114     OK
  TCGA-KIRC       DeepMLP-A   0.7216   0.7535  0.0319  WATCH
  TCGA-KIRC    ShallowMLP-B   0.7161   0.7795  0.0634  WATCH
  TCGA-LUAD       DeepMLP-A   0.5739   0.6395  0.0656  WATCH
  TCGA-LUAD    ShallowMLP-B   0.5524   0.6221  0.0697  WATCH
  OK gap < 0.03 = stable | WATCH gap < 0.08 | LEAK gap >= 0.08 = risk

  NB00 -> NB03 delta (Option A test C-index)
  Cohort           NB00    NB03-A     Delta
  --------------------------------------------
  TCGA-LGG          n/a    0.8240       n/a --
  TCGA-KIRC         n/a    0.7535       n/a --
  TCGA-LUAD         n/a    0.6395       n/a --

  Saved: D:\JupyterWork\NB04\results\NB03\NB03_benchmark_summary.csv

  Outputs:
    results/NB03/figures/NB03_lgg_